**GEOG5415M Programming for Spatial Data Science**

# Week 7: Machine Learning in Action

In this practical we will go through a worked example of machine learning in action. We are going to replicate some of the work in the paper:

 - Asher, M., Oswald, Y., & Malleson, N. (2025). Understanding pedestrian dynamics using machine learning with real-time urban sensors. _Environment and Planning B: Urban Analytics and City Science_, 52(8), 1994-2017. DOI:[10.1177/23998083251319058](https://doi.org/10.1177/23998083251319058)

The aim of the work is to build a model that can predict _pedestrian footfall_ for a particular hour, given information about the time, the weather, and the built environment.

If you are interested, the original code, in full, is available from the paper's Github repository: 

 - [github.com/nickmalleson/footfall/tree/main/MelbourneAnalysis](https://github.com/nickmalleson/footfall/tree/main/MelbourneAnalysis)

We will go through the following steps:
 1. Data preparation (including downloading, cleaning and linking)
 1. Data analysis (look for missing data and look at data distributions)
 1. Model the data (including model selection)

In [ ]:
# import required packages

# XXXX 

import geopandas as gpd
import pandas as pd
import seaborn as sns
from scipy import stats
import numpy as np

import matplotlib.pyplot as plt

# import the required machine learning packages
from sklearn import cluster
from sklearn.preprocessing import scale

# set seaborn plotting theme to white
sns.set_theme(style="white")

# Data Preparation

We need to download, process, clean and merge the following data sources:

 - footfall counts (our target)
 - weather conditions
 - built environment features
 - date information (public holidays, school term times, etc,)

If you are interested, the full code is organised into various notebooks in the original repositoy's [PreparingData](https://github.com/nickmalleson/footfall/tree/main/MelbourneAnalysis/1.%20PreparingData) directory.

## Downloading and reading data

### Footfall counts

The first dataset we need is the footfall counts -- i.e. the counts of pedestrians who pass Melbourne's footfall cameras every hour. In the paper, we had to find the data on the Melbourne Open Data Portal, download a few different files (one for old counts, one for new ones) and merge them. If you want to see the code to do this in full, have a look at the [Data Preparation Folder](https://github.com/nickmalleson/footfall/tree/main/MelbourneAnalysis/1.%20PreparingData) in the paper's repository. For this practical, we have made the data available with the notebook to save some time, but there is still a lot of cleaning and preparation to do.

The data are stored in the file called "sensor_counts.csv.gz" in the data/week_7 directory. Note that the file has the '.gz' extension. This is short for 'gzip' and it means that the csv file has been compressed using an algorithm called 'gzip' so that it takes up less space. Fortunately pandas makes it really easy to read and write files using the argument `compression='gzip'`. As with other data, we can read it directly from the module's github repository

<font color='orchid'> <b>Run the code below to read the compressed csv file and assign it to a variable called `sensor_counts`</b></font>.

In [21]:
url_to_sensor_data = "https://github.com/MSc-Urban-Environmental-Leeds/GEOG5415M-Programming-for-Spatial-Data-Science/raw/refs/heads/main/data/week_7/sensor_counts.csv.gz"
sensor_counts = pd.read_csv(url_to_sensor_data, compression='gzip')

Have a look at the sensor counts. Each row holds the number of counts from a sensor for a particular hour (the `hourly_counts`) column, as well as some other information

In [13]:
sensor_counts

,Unnamed: 0,ID,datetime,year,month,mdate,day,time,sensor_id,Sensor_Name,hourly_counts
0,0,421820211114,2021-11-14 18:00:00,2021,November,14,Sunday,18,42,UM1_T,49
1,1,461320231124,2023-11-24 13:00:00,2023,November,24,Friday,13,46,Pel147_T,189
2,2,25420220305,2022-03-05 04:00:00,2022,March,5,Saturday,4,25,MCEC_T,42
3,3,30320240804,2024-08-04 03:00:00,2024,August,4,Sunday,3,30,Lon189_T,344
4,4,751120240108,2024-01-08 11:00:00,2024,January,8,Monday,11,75,SprFli_T,43
...,...,...,...,...,...,...,...,...,...,...,...
1850388,1850388,631320230527,2023-05-27 13:00:00,2023,May,27,Saturday,13,63,Bou231_T,1052
1850389,1850389,501620240323,2024-03-23 16:00:00,2024,March,23,Saturday,16,50,Lyg309_T,557
1850390,1850390,201620240417,2024-04-17 16:00:00,2024,April,17,Wednesday,16,20,LtB170_T,478
1850391,1850391,621520230905,2023-09-05 15:00:00,2023,September,5,Tuesday,15,62,Lat224_T,382


There are some columns that we don't need so lets get rid of them. The easiest way to do this is to use the `sensor_counts.drop()` function. For example, if you wanted to remove a column called 'MyColumn' you could remove it with:
```python
sensor_counts.drop(colums = ['MyColumn'])
```

<font color='orchid'> <b>Edit the code below remove the columns called 'ID', 'Sensor_Name' and 'Unnamed: 0'.</b></font>.

In [26]:
sensor_counts = sensor_counts.drop(columns = ['ID', 'Sensor_Name', 'Unnamed: 0'])

The next chunk will check that the column removal worked correctly. If it runs without an error then you have done it right.
Ask someone if you'd like us to explain how it works)

In [28]:
columns_still_in_df = [c for c in ['ID', 'Sensor_Name', 'Unnamed: 0'] if c in sensor_counts.columns]
if columns_still_in_df:
    raise ValueError(f"These columns are still in the dataset: {columns_still_in_df}")
print("All columns removed successfully")

All columns removed successfully


Check for duplicates

XXXX

In [31]:
sensor_counts[sensor_counts.duplicated(subset=['datetime', 'sensor_id'])]

,datetime,year,month,mdate,day,time,sensor_id,hourly_counts


### Sensor locations

We know what the counts at each sensor are, but we don't know _where_ the sensors are located. We need to know this so that we can map the sensor counts, and also so that we can join the sensors to other spatial data about the local built environment.

In [ ]:
sensor_locations = pd.read_csv("sensor_locations.csv")

XXXX HERE

# Data Analysis

# Modelling